In [1]:
from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph, START

from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.tools import tool

from pydantic import BaseModel, Field

from langchain_experimental.tools import PythonREPLTool

import wikipedia
from langchain_community.tools import WikipediaQueryRun, RequestsGetTool
from langchain_community.utilities import WikipediaAPIWrapper, TextRequestsWrapper
from bs4 import BeautifulSoup

from datetime import date, datetime

import requests

C:\Users\bjit\AppData\Local\Temp\ipykernel_22100\3068892426.py:15: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
C:\Users\bjit\AppData\Local\Temp\ipykernel_22100\3068892426.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun, RequestsGetTool


In [2]:
llm = ChatOllama(model="qwen3:8b")

In [3]:
class PythonREPLInput(BaseModel):
    query: str = Field(description="Valid Python code to execute)")

python_repl = PythonREPLTool(args_schema=PythonREPLInput)

In [4]:
wikipedia.set_user_agent("MyLangChainApp")
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [5]:
requests_wrapper = TextRequestsWrapper()
requests_tool = RequestsGetTool(requests_wrapper=requests_wrapper, allow_dangerous_requests=True)

In [6]:
@tool
def age_calculator(birth_date: str) -> int:
    """Age calculating function, From current time it substract birth date, input in format DD-MM-YYYY"""
    print("Tool used: age_calculator")
    dob = datetime.strptime(birth_date, "%d-%m-%Y").date()
    today = date.today()
    age = today.year - dob.year
    if (today.month, today.day) < (dob.month, dob.day):
        age -= 1
    return int(age)

In [7]:
@tool
def get_weather(city: str) -> str:
    """Get the current temperature for a given city."""
    print("Tool used: get_weather")
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
    geo_data = requests.get(geo_url).json()

    lat = geo_data["results"][0]["latitude"]
    lon = geo_data["results"][0]["longitude"]

    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    weather_data = requests.get(weather_url).json()

    temp = weather_data["current_weather"]["temperature"]
    return f"The current temperature in {city} is {temp}°C"

In [8]:
tools = [python_repl, wiki_tool, requests_tool, age_calculator, get_weather]
llm_with_tools = llm.bind_tools(tools)

In [9]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [10]:
def chat_node(state: ChatState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [11]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

In [12]:
agent = graph.compile()

In [13]:
prompt = "what is 4 + 4?"
out = agent.invoke({'messages':[HumanMessage(content=prompt)]})
print(out['messages'][-1].content)
print([m.name for m in out['messages'] if m.type == "tool"])

Python REPL can execute arbitrary code. Use with caution.


The result of 4 + 4 is 8.
['Python_REPL']


In [14]:
query = """
1. What is the current temperature in Dhaka?
2. Calculate the age of someone born on 15-06-1995.
3. Search Wikipedia for 'Python programming language' and give a short summary.
4. Use Python to calculate 25 * 4.
5. Fetch the content of https://books.toscrape.com/
"""

out = agent.invoke({"messages": [HumanMessage(content=query)]})
print(out["messages"][-1].content)


Tool used: get_weatherTool used: age_calculator

The provided HTML code represents a **product listing page** for an online bookstore, likely built using **Bootstrap 3** and the **Oscar** e-commerce platform. Here's a breakdown of its structure and functionality:

---

### **1. Layout & Structure**
- **Bootstrap Grid System**: 
  - Products are organized in a responsive grid using classes like `col-xs-6`, `col-sm-4`, `col-md-3`, and `col-lg-3`. This ensures the layout adapts to different screen sizes.
  - Each product is wrapped in an `<article class="product_pod">` element.

---

### **2. Product Details**
Each product includes:
- **Image**: 
  - A thumbnail image (`<img>`) linked to the product page (e.g., `catalogue/olio_984/index.html`).
- **Star Ratings**: 
  - Custom icons (likely from **Font Awesome**) for ratings (e.g., `icon-star`). The number of stars corresponds to the rating (e.g., `One` for 1 star, `Two` for 2 stars).
- **Title**: 
  - A link to the product page (`<a href=

In [15]:
print([m.name for m in out['messages'] if m.type == "tool"])

['get_weather', 'age_calculator', 'wikipedia', 'Python_REPL', 'requests_get']
